In [5]:
#!/usr/bin/env python3
"""
b=3 block RG fast-mode computation (2 fast modes per block), periodic.

Computes:
  c1(3) = (1/3) * (1/2π) ∫_{-π}^{π} tr[K3(k)^{-1}] dk
  m_eff^2 = κ * c1(3)

Notebook/Colab safe (ignores injected args).
"""

from __future__ import annotations
import argparse
import json
import numpy as np


def parse_args_safe():
    ap = argparse.ArgumentParser(add_help=True)
    ap.add_argument("--Nk", type=int, default=4096)
    ap.add_argument("--kappa", type=float, default=0.1)
    ap.add_argument("--out", type=str, default="rg_fastmode_mass_b3_true.json")
    args, _ = ap.parse_known_args()
    return args


# Orthonormal b=3 block basis (rows are basis vectors in site coords)
# e0 = (1,1,1)/sqrt(3)      (coarse)
# e1 = (1,-1,0)/sqrt(2)     (fast)
# e2 = (1,1,-2)/sqrt(6)     (fast)
E = np.array(
    [
        [1 / np.sqrt(3), 1 / np.sqrt(3), 1 / np.sqrt(3)],
        [1 / np.sqrt(2), -1 / np.sqrt(2), 0.0],
        [1 / np.sqrt(6), 1 / np.sqrt(6), -2 / np.sqrt(6)],
    ],
    dtype=np.complex128,
)


def K3_symbol(k: float) -> np.ndarray:
    """
    2x2 fast-mode block-circulant symbol K3(k) for nearest-neighbor kinetic term
    with inter-block bond between site 2 of block n and site 0 of block n+1.

    Returns: 2x2 complex Hermitian matrix.
    """
    phase = np.exp(1j * k)

    # Site-space Laplacian contribution for a 3-site block with one outgoing bond.
    # Internal bonds: 0-1, 1-2
    # Inter-block bond: 2 --(phase)--> 0(next)
    K_site = np.zeros((3, 3), dtype=np.complex128)

    # Internal bonds (0-1, 1-2): Laplacian pieces
    K_site[0, 0] += 1
    K_site[1, 1] += 2
    K_site[2, 2] += 1
    K_site[0, 1] -= 1
    K_site[1, 0] -= 1
    K_site[1, 2] -= 1
    K_site[2, 1] -= 1

    # Inter-block bond (2 to next 0): adds to diag and phase-coupled off-diag
    K_site[2, 2] += 1
    K_site[0, 0] += 1
    K_site[2, 0] -= np.conjugate(phase)
    K_site[0, 2] -= phase

    # Transform to block basis
    K_block = E @ K_site @ E.conjugate().T

    # Fast-fast subblock (indices 1,2)
    return K_block[1:, 1:]


def compute_c1_b3(Nk: int) -> float:
    ks = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
    s = 0.0
    for k in ks:
        Kk = K3_symbol(float(k))
        Kk_inv = np.linalg.inv(Kk)
        s += float(np.trace(Kk_inv).real)
    avg = s / Nk
    integral_over_2pi = avg / (2.0 * np.pi)
    return (1.0 / 3.0) * integral_over_2pi


def main():
    args = parse_args_safe()
    Nk = int(args.Nk)
    kappa = float(args.kappa)

    c1 = compute_c1_b3(Nk)
    m_eff2 = c1 * kappa

    print(f"Nk = {Nk}")
    print(f"κ  = {kappa}")
    print("")
    print("TRUE b=3 results:")
    print(f"  c1(3)   = {c1:.15g}")
    print(f"  m_eff^2 = {m_eff2:.15g}")
    print("")
    print(f"saved: {args.out}")

    out = {"block_size": 3, "Nk": Nk, "kappa": kappa, "c1": c1, "m_eff2": m_eff2}
    with open(args.out, "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2)


if __name__ == "__main__":
    main()

Nk = 4096
κ  = 0.1

TRUE b=3 results:
  c1(3)   = 0.0478945612083242
  m_eff^2 = 0.00478945612083242

saved: rg_fastmode_mass_b3_true.json
